<!-- VIDEO: 1 / LangChain v1.0 — 왜 필요한가 + init_chat_model -->

# 02. LangChain Core + LCEL

> **학습 목표**
> 1. LangChain v1.0의 단순화된 네임스페이스(`langchain.messages`, `langchain.chat_models` 등)의 구조를 이해한다.
> 2. `init_chat_model`을 사용해 모델 식별자만 변경하여 OpenAI, Gemini, Groq, Ollama 간 전환을 수행한다.
> 3. **Messages → Prompt Template → Output Parser** 의 표준 처리 흐름을 익힌다.
> 4. **LCEL**의 `|` 파이프 연산자로 체인을 선언적으로 구성한다.
> 5. `RunnableParallel`, `RunnablePassthrough`, `RunnableLambda`로 분기, 병렬 실행, 전후처리를 구현한다.
>
> **선수 지식**: 01번 노트북(LLM 호출 기본 개념).

---

> **LangChain v1.0의 변경점**
> - `langchain` 네임스페이스가 5개로 단순화됨: `langchain.messages`, `langchain.tools`, `langchain.agents`, `langchain.chat_models`, `langchain.embeddings`.
> - `LLMChain`, 전통 Retriever 등 레거시 기능은 `langchain-classic` 패키지로 분리됨.
> - 본 강의는 v1.0 이상의 API만 사용합니다.

01번 노트북에서는 OpenAI SDK 인터페이스에 코드가 결합되어 있었습니다. 본 노트북부터는 `init_chat_model("gpt-4.1-mini")`와 `init_chat_model("gemini-2.5-flash-lite")`가 동일한 인터페이스를 제공한다는 점을 활용해, 모델 교체에 따른 코드 수정을 최소화하는 방법을 학습합니다.

---

## 1. 환경 설정 — LLM 선택

본 강의에서 사용하는 표준 LLM 선택 패턴입니다. 아래 옵션 중 한 가지 키만 보유하면 진행할 수 있으며, 무료 옵션을 우선 권장합니다.

```
[권장 순서]
1순위: Gemini 2.5 Flash-Lite (무료, 결제 등록 불필요, 한국어 우수)
2순위: Groq Qwen3 32B       (무료, 빠른 추론 속도, 한국어 우수)
3순위: Ollama Gemma 4 E4B   (무료, 로컬 실행, RAM 16GB 이상 권장)
선택:  OpenAI gpt-4.1-mini   (유료, 결제 등록 필요)
```

In [1]:
from dotenv import load_dotenv
load_dotenv()

import warnings
warnings.filterwarnings("ignore")

import os
print("✅ API Key 감지:",
      "Gemini" if os.getenv("GOOGLE_API_KEY") else "—",
      "Groq" if os.getenv("GROQ_API_KEY") else "—",
      "OpenAI" if os.getenv("OPENAI_API_KEY") else "—")

✅ API Key 감지: Gemini Groq OpenAI


## 2. Models — `init_chat_model`

LangChain v1.0에서는 `init_chat_model("provider:model")` 한 함수로 모든 제공자의 모델을 통합 초기화할 수 있습니다. 모델 문자열만 변경하면 다른 제공자로 전환되며, 호출 인터페이스는 동일하게 유지됩니다.

In [25]:
from langchain.chat_models import init_chat_model

# ✅ Option 1: Google Gemini 2.5 Flash-Lite (무료, 카드 등록 X)
#llm = init_chat_model("google_genai:gemini-2.5-flash-lite", temperature=0.3)

# ✅ Option 2: Groq Qwen3 32B (무료, 한국어 강함, 매우 빠름)
llm = init_chat_model(model_provider="groq", model="qwen/qwen3-32b", temperature=0.3)

# ✅ Option 3: Ollama Gemma 4 E4B (완전 무료, 로컬)
# llm = init_chat_model("ollama:gemma4:e4b", temperature=0.3)

# ⚙️ Option 4: OpenAI (유료)
#llm = init_chat_model(model_provider="openai", model="gpt-4.1-mini", temperature=0.3)

print(llm)
print(llm.model_name)

output_version=None profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x0000022BFB450560> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022BFB452840> model_name='qwen/qwen3-32b' temperature=0.3 model_kwargs={} groq_api_key=SecretStr('**********') groq_api_base=None groq_proxy=None
qwen/qwen3-32b


In [4]:
# 가장 간단한 호출: 문자열 직접 전달
response = llm.invoke("LangChain v1.0 의 핵심 철학을 한 문장으로 설명하세요.")
print(response.content)

<think>
Okay, I need to explain the core philosophy of LangChain v1.0 in one sentence. Let me start by recalling what LangChain is. It's a framework for building applications with large language models. The user wants the key philosophy, so I should focus on the main idea that LangChain is built around.

I remember that LangChain emphasizes modularity and composability. That means allowing developers to combine different components like models, prompts, and memory in flexible ways. Also, there's a focus on making it easy to integrate with various LLMs and other tools. Maybe something about enabling the creation of complex workflows by chaining together simple building blocks.

Wait, the user mentioned v1.0 specifically. I should check if there are any version-specific features. The core philosophy probably hasn't changed much, but maybe in v1.0 they solidified certain aspects. The key points are modularity, composability, and enabling developers to build end-to-end applications with LL

## 3. Messages — 대화의 기본 단위

v1.0에서는 메시지 타입이 `langchain.messages` 모듈에 통합되어 있습니다. 메시지는 `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage` 등의 클래스로 표현되며, 모델 호출 시 이들의 리스트를 입력으로 사용합니다.

In [5]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage

messages = [
    SystemMessage(content="당신은 사내 IT 도우미입니다. 간결하게 답하세요."),
    HumanMessage(content="VPN 연결이 자꾸 끊기는데 어떻게 해야 하나요?"),
]
reply = llm.invoke(messages)
print(reply.content)
print("\n메시지 타입:", type(reply).__name__)  # AIMessage

<think>
Okay, the user is having trouble with their VPN connection dropping. Let me think about the possible causes and solutions. First, I should check the basics. Maybe their internet connection is unstable. They could try restarting the router or modem. Also, checking if other devices are working on the same network might help identify if it's a local issue.

Next, the VPN client itself might be the problem. Updating the client software could resolve compatibility issues. Sometimes, firewall or antivirus programs block the connection, so temporarily disabling them might help. They should also check if there are any recent updates on their operating system that could be causing conflicts.

Another angle is the server side. Maybe the company's VPN server is overloaded or having issues. They could contact IT support to check the server status. If it's a configuration problem, resetting the VPN settings or reconfiguring the client might work. Also, switching between different protocols 

In [6]:
# Key만 확인
print(list(reply.model_dump().keys()))

['content', 'additional_kwargs', 'response_metadata', 'type', 'name', 'id', 'tool_calls', 'invalid_tool_calls', 'usage_metadata']


In [7]:
# 키 + 값 함께 확인
for key, value in reply.model_dump().items():
    print(f"{key} => {value}")

content => <think>
Okay, the user is having trouble with their VPN connection dropping. Let me think about the possible causes and solutions. First, I should check the basics. Maybe their internet connection is unstable. They could try restarting the router or modem. Also, checking if other devices are working on the same network might help identify if it's a local issue.

Next, the VPN client itself might be the problem. Updating the client software could resolve compatibility issues. Sometimes, firewall or antivirus programs block the connection, so temporarily disabling them might help. They should also check if there are any recent updates on their operating system that could be causing conflicts.

Another angle is the server side. Maybe the company's VPN server is overloaded or having issues. They could contact IT support to check the server status. If it's a configuration problem, resetting the VPN settings or reconfiguring the client might work. Also, switching between different

### AIMessage의 메타데이터 활용

`AIMessage` 객체에는 응답 본문 외에도 토큰 사용량, 모델 식별자, 종료 사유 등의 메타데이터가 포함되어 있습니다. 비용 추적과 디버깅에 유용합니다.

In [8]:
print("content:", reply.content[:80], "...")
print("usage_metadata:", reply.usage_metadata)  # 토큰 사용량
print("response_metadata.model:", reply.response_metadata.get("model_name"))

content: <think>
Okay, the user is having trouble with their VPN connection dropping. Let ...
usage_metadata: {'input_tokens': 47, 'output_tokens': 464, 'total_tokens': 511}
response_metadata.model: qwen/qwen3-32b


## 4. Prompt Template — 변수 기반 프롬프트

동일한 프롬프트 패턴을 여러 입력에 재사용할 때 사용합니다. `ChatPromptTemplate`은 시스템 메시지와 사용자 메시지의 템플릿을 정의하고, 변수 치환을 통해 일관된 형태의 메시지 리스트를 생성합니다.

In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 입니다. 친절하고 전문적으로 답변하세요."),
    ("human", "{question}"),
])

# 변수 채우기
filled = prompt.invoke({
    "role": "ABC 사내 HR 도우미",
    "question": "연차 사용은 며칠 전까지 신청해야 하나요?",
})
print(type(filled))
print(filled)

<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[SystemMessage(content='당신은 ABC 사내 HR 도우미 입니다. 친절하고 전문적으로 답변하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content='연차 사용은 며칠 전까지 신청해야 하나요?', additional_kwargs={}, response_metadata={})]


In [10]:
for msg in filled.messages:
    msg.pretty_print()

================================ System Message ================================

당신은 ABC 사내 HR 도우미 입니다. 친절하고 전문적으로 답변하세요.
================================ Human Message =================================

연차 사용은 며칠 전까지 신청해야 하나요?


## 5. Output Parser — 응답 변환

`StrOutputParser`는 가장 단순한 형태의 파서로, `AIMessage` 객체에서 문자열 `content`만 추출합니다. 이후 절에서 다룰 LCEL 체인의 마지막 단계에 자주 배치되어, 후속 코드가 다루기 쉬운 일반 문자열을 반환하도록 합니다.

In [12]:
from langchain_core.output_parsers import StrOutputParser

print(type(reply))
parser = StrOutputParser()
text = parser.invoke(reply)   # AIMessage → str
print(type(text), "\n", text[:80], "...")

<class 'langchain_core.messages.ai.AIMessage'>
<class 'langchain_core.messages.base.TextAccessor'> 
 <think>
Okay, the user is having trouble with their VPN connection dropping. Let ...


<!-- VIDEO: 2 / LCEL 파이프 — 한 줄로 끝나는 체인 -->

## 6. LCEL — 파이프 연산자 기반의 체인 구성

**LCEL(LangChain Expression Language)** 은 `|` 파이프 연산자를 사용해 컴포넌트를 연결하고, 체인을 **선언적으로 정의**하는 표현 방식입니다.

### 비유 1: 유닉스 파이프

`cat file | grep ERROR | wc -l`처럼, 왼쪽 컴포넌트의 출력이 오른쪽 컴포넌트의 입력으로 전달되는 단방향 흐름과 동일한 철학을 따릅니다.

```python
prompt | llm | parser
#  │       │      └── AIMessage → str (문자열 추출)
#  │       └────────── 채워진 프롬프트 → AIMessage (LLM 호출)
#  └────────────────── 변수 dict → 채워진 프롬프트 (Template 적용)
```

### 비유 2: 공정 라인

원자재(입력 변수) → 공정 1(프롬프트 템플릿 적용) → 공정 2(LLM 추론) → 공정 3(파서) → 결과물(문자열)의 흐름입니다. 각 공정은 독립적이므로 한 단계만 다른 컴포넌트로 교체해도 전체 라인은 그대로 동작합니다(예: `StrOutputParser`만 JSON 파서로 교체).

### 다이어그램

```mermaid
flowchart LR
    IN(["📥 dict input<br/>{role, question}"]):::input
    PT["📝 Prompt<br/>ChatPromptTemplate<br/>{role} · {question}"]:::node
    LM["🤖 LLM<br/>init_chat_model<br/>→ AIMessage"]:::node
    PA["🔍 Parser<br/>StrOutputParser<br/>→ str"]:::node
    OUT(["📤 최종 문자열"]):::output

    IN --> PT --> LM --> PA --> OUT

    classDef input  fill:#4f46e5,stroke:#3730a3,color:#fff
    classDef node   fill:#1e293b,stroke:#475569,color:#e2e8f0
    classDef output fill:#059669,stroke:#047857,color:#fff
```

코드 표현은 다음과 같이 간결합니다.

```
prompt | llm | parser
```

### 파이프 연산자의 이점

1. **가독성**: "프롬프트 → 모델 → 파서"의 처리 흐름이 코드에 그대로 드러납니다.
2. **재사용성**: `prompt | llm` 부분을 변수로 분리하면 서로 다른 파서와 조합할 수 있습니다.
3. **공통 인터페이스**: 각 Runnable은 `.invoke()`, `.stream()`, `.batch()`를 자동 지원합니다.

In [13]:
chain = prompt | llm | parser

# 한 번의 invoke 로 prompt → llm → parser 전체가 실행됨
answer = chain.invoke({
    "role": "ABC HR 도우미",
    "question": "연차는 언제까지 신청해야 하나요?",
})
print(answer)

<think>
Okay, the user is asking when they need to apply for their annual leave. Let me check the company's policies. From what I remember, ABC Company usually requires employees to submit their annual leave applications by a certain date each year. I think it's around the end of December for the following year's leave. Wait, but maybe it's different depending on the department or role? Hmm, no, I believe the general policy is the same across the company.

Wait, let me confirm. The HR guidelines mention that employees should apply for their annual leave by December 31st of the current year for the next year's schedule. That way, the HR team can plan and allocate resources properly. Also, if there are any changes or special circumstances, employees should notify their manager and HR as soon as possible. Oh, and if someone hasn't applied by the deadline, they might need to apply for the remaining days during the year, but that's subject to approval based on workload. 

I should also ment

### 스트리밍 호출

LLM은 토큰 단위로 응답을 생성하므로, `.stream()`을 사용하면 토큰이 생성되는 즉시 클라이언트로 전달받을 수 있습니다. 사용자 응답 지연 체감을 줄이는 데 유용합니다.

In [14]:
for chunk in chain.stream({
    "role": "ABC IT 도우미",
    "question": "비밀번호 정책을 한 문단으로 알려주세요.",
}):
    print(chunk, end="", flush=True)
print()

<think>
Okay, the user is asking for a password policy in a single paragraph. Let me start by recalling the key elements of a strong password policy. First, password length is important—usually between 12 to 16 characters. Then, complexity requirements: a mix of uppercase, lowercase, numbers, and symbols. They should avoid common words or patterns. Also, regular updates, maybe every 90 days. Multi-factor authentication is a good addition. Account lockout after too many failed attempts. Maybe mention not reusing passwords and using a password manager. Let me make sure it's all in one paragraph without being too technical. Need to keep it concise but comprehensive. Check for clarity and flow. Alright, that should cover the essentials.
</think>

ABC IT의 비밀번호 정책은 보안 강화를 위해 최소 12자 이상의 길이를 요구하며, 대소문자, 숫자, 특수기호를 조합하여 구성해야 합니다. 사용자 인증 시 3회 이상 실패 시 계정 잠금이 발생하며, 90일 주기로 비밀번호 변경이 의무화됩니다. 공통 패턴(예: "1234", "password") 또는 개인 정보(예: 생년월일)를 포함한 약한 비밀번호는 금지되며, 다중 인증(MFA)을 권장합니다. 추가로, 동일한 비밀번호 재사용을 방지하고,

### 배치 호출

`.batch()`를 사용하면 여러 입력을 한 번의 호출 단위로 묶어 병렬 처리할 수 있습니다. 다수의 독립적인 입력을 처리할 때 전체 처리 시간을 단축할 수 있습니다.

In [11]:
results = chain.batch([
    {"role": "HR 도우미", "question": "경조사 휴가 일수를 알려주세요."},
    {"role": "IT 도우미", "question": "VPN 접속 방법은?"},
    {"role": "재무 도우미", "question": "출장비는 얼마까지 지원되나요?"},
])
for i, r in enumerate(results, 1):
    print(f"[{i}] {r[:60]}...")

[1] 안녕하세요! 경조사 휴가 일수에 대해 문의주셨군요.

경조사 휴가 일수는 회사의 내부 규정이나 근로기준법에 ...
[2] 안녕하세요! VPN 접속 방법에 대해 궁금하시군요. VPN 접속은 사용하시는 기기(PC, 스마트폰 등)와 V...
[3] 안녕하세요! 출장비 지원에 대해 궁금하시군요.

출장비 지원 금액은 회사의 정책에 따라 다를 수 있습니다. ...


## 7. Runnable 컴포넌트

LCEL 체인을 구성하는 모든 부품은 공통적으로 `Runnable` 인터페이스를 따릅니다. 보다 복잡한 처리 흐름을 구성할 때 다음 유틸리티 컴포넌트를 활용합니다.

| 컴포넌트 | 역할 |
|---|---|
| `RunnableSequence` | 파이프라인 (`\|` 연산자로 자동 생성) |
| `RunnableParallel` | 여러 체인을 병렬 실행 |
| `RunnablePassthrough` | 입력을 그대로 전달 (딕셔너리 필드 채울 때) |
| `RunnableLambda` | 일반 파이썬 함수를 Runnable로 래핑 |

### 7.1 `RunnablePassthrough` + `RunnableParallel`

동일한 입력에 대해 **여러 관점의 응답을 동시에** 생성하는 패턴입니다. 각 분기 체인이 독립적으로 실행되므로 직렬 호출 대비 응답 시간을 단축할 수 있습니다.

In [32]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

llm = init_chat_model(model_provider="groq", model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0.3)

pro_prompt  = ChatPromptTemplate.from_messages([("system", "긍정적 관점으로 답하세요."), ("human", "{q}")])
con_prompt  = ChatPromptTemplate.from_messages([("system", "비판적 관점으로 답하세요."), ("human", "{q}")])

pros_chain = pro_prompt | llm | parser
cons_chain = con_prompt | llm | parser

debate = RunnableParallel(
    question=RunnablePassthrough(),   # 입력의 q를 그대로 통과
    pros=pros_chain,
    cons=cons_chain,
)

out = debate.invoke({"q": "재택근무를 전면 도입하는 것은 좋은가?"})

print(type(out))
print(out.keys())
print("긍정적인 관점:")
print("PROS:", out["pros"], "...")


<class 'dict'>
dict_keys(['question', 'pros', 'cons'])
긍정적인 관점:
PROS: 재택근무를 전면 도입하는 것은 여러 가지 장점이 있습니다. 

첫째, 직원들의 일과 삶의 균형을 개선할 수 있습니다. 

둘째, 출근하지 않아도 되기 때문에 출퇴근 시간과 교통비를 절약할 수 있습니다.

셋째, 보다 유연한 근무 환경을 제공할 수 있습니다.

하지만, 재택근무를 전면 도입하는 경우 발생할 수 있는 문제점도 고려해야 합니다. 예를 들어, 팀원들 간의 소통과 협력이 어려워질 수 있고, 보안 문제나 책임감이 떨어질 수 있습니다. 따라서 재택근무를 전면 도입하기 전에 이러한 문제점들을 고려하여 적절한 대책을 마련하는 것이 중요합니다. ...


In [31]:
print("비판적인 관점:")
print("CONS:", out["cons"], "...")

비판적인 관점:
CONS: 재택근무를 전면 도입하는 것에 대한 장단점을 살펴보겠습니다.

장점:

*   **유연성 및 자율성**: 재택근무는 직원들에게 더 많은 유연성과 자율성을 제공합니다. 직원들은 자신의 일과 생활의 균형을 맞추기 위해 더 많은 선택권을 가질 수 있습니다.
*   **생산성 향상**: 일부 직원들은 재택근무를 통해 더 집중해서 일할 수 있고, 생산성이 향상될 수 있습니다. 
*   **비용 절감**: 재택근무는 회사와 직원 모두에게 비용 절감 효과를 가져올 수 있습니다. 회사는 사무실 공간을 줄일 수 있고, 직원은 통근 비용을 절감할 수 있습니다.

단점:

*   **소통 및 협업의 어려움**: 재택근무는 직원들 간의 소통과 협업을 어렵게 만들 수 있습니다. 얼굴을 마주하지 않고 일하기 때문에 의사소통의 부재로 인한 문제가 생길 수 있습니다.
*   **감시와 평가의 어려움**: 관리자가 직원들의 성과를 제대로 평가하는 것이 어려울 수 있습니다. 
*   **외로움과 고립**: 재택근무는 직원들이 외로움과 고립감을 느낄 수 있습니다. 지속적인 상호작용이 없기 때문에 동료들과의 관계가 약해질 수 있습니다.

결론:

재택근무를 전면 도입하는 것은 장단점이 있습니다. 회사와 직원들은 재택근무의 장단점을 고려하여 자신들에게 적합한 방식을 찾아야 합니다. 또한, 재택근무를 성공적으로 시행하기 위해서는 명확한 목표와 지침을 설정하고, 직원들의 소통과 협력을 지원하기 위한 시스템을 구축하는 것이 중요합니다. ...


### 7.2 `RunnableLambda` — 파이썬 함수 삽입

체인 중간에 전처리 또는 후처리 단계를 추가할 때 사용합니다. 일반 파이썬 함수를 Runnable로 래핑해 LCEL 체인에 자연스럽게 결합할 수 있습니다.

In [33]:
from langchain_core.runnables import RunnableLambda

def shorten(text: str, max_len: int = 80) -> str:
    return text if len(text) <= max_len else text[:max_len] + " ..."

# 결과를 짧게 자르는 체인
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "한 문장으로 요약하세요."),
    ("human", "{text}"),
])

chain2 = summary_prompt | llm | parser | RunnableLambda(shorten)
print(chain2.invoke({"text": "LangChain 은 LLM 기반 애플리케이션을 만들기 위한 프레임워크로, 프롬프트 관리, 메모리, 도구 호출, 에이전트 실행, RAG 등 여러 기능을 제공합니다."}))

LangChain은 대규모 언어 모델(LLM) 기반 애플리케이션을 구축하기 위한 프레임워크입니다.


## 8. 멀티턴 대화를 위한 메시지 누적

간단한 대화 흐름은 메시지 리스트를 직접 관리하는 방식으로 구현할 수 있습니다. 보다 복잡한 메모리 관리(체크포인팅, 세션 단위 상태 저장 등)는 LangGraph의 `MemorySaver` 체크포인터에서 별도로 다룹니다.

In [14]:
history = [SystemMessage(content="당신은 ABC IT 도우미입니다.")]

def chat(user_msg: str) -> str:
    history.append(HumanMessage(content=user_msg))
    ai = llm.invoke(history)
    history.append(ai)
    return ai.content

print("User:", "안녕, 내 이름은 지훈이야.")
print("Bot :", chat("안녕, 내 이름은 지훈이야."))
print()
print("User:", "내 이름을 기억해?")
print("Bot :", chat("내 이름을 기억해?"))

User: 안녕, 내 이름은 지훈이야.
Bot : 안녕하세요, 지훈님! ABC IT 도우미입니다. 무엇을 도와드릴까요?

User: 내 이름을 기억해?
Bot : 네, 지훈님! 방금 알려주신 이름을 기억하고 있습니다. 앞으로도 지훈님이라고 불러드릴게요. 😊


<!-- VIDEO: 3 / 한국어 실전 LCEL — 메뉴 추천 + 뉴스 헤드라인 병렬 처리 -->

---

## 9. 한국어 비즈니스 시나리오 실습

### 9.1 메뉴 추천 체인

LCEL의 강점은 **여러 단계의 처리 흐름을 한 줄로 정리**할 수 있다는 점입니다. 사용자 취향 입력 → 메뉴 추천 → 가독성 있는 출력 형식까지 이어지는 체인을 구성합니다.

In [34]:
menu_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 한국 음식점 추천 전문가입니다. "
               "사용자의 기분·날씨·예산을 보고 가장 잘 어울리는 메뉴 3개를 추천하세요. "
               "각 메뉴마다 이모지 1개와 한 줄 추천 이유를 붙이세요."),
    ("human", "기분: {mood}\n날씨: {weather}\n예산: {budget}원"),
])

menu_chain = menu_prompt | llm | parser

print(menu_chain.invoke({"mood": "축 처지는 월요일", "weather": "비 오고 쌀쌀함", "budget": "12000"}))

월요일에 기분이 축 처진다면, 따뜻하고 얼큰한 국물 요리가 딱이죠! 비가 오는 쌀쌀한 날씨에 어울리는 메뉴를 추천해 드리겠습니다.

1.  **김치찌개** 🍲 - 얼큰한 김치찌개는 쌀쌀한 날씨에 딱입니다.
2.  **삼계탕** 🍗 - 삼계탕은 기력 회복에 도움이 되는 메뉴입니다.
3.  **해물짬뽕** 🍜 - 해물짬뽕은 얼큰한 국물에 해물이 가득 들어 있어 월요일에 기분 전환하기 좋습니다.

예산 12000원으로 위 메뉴를 모두 주문할 수 있습니다.


### 9.2 뉴스 헤드라인 — 요약과 감정 분석 병렬 처리

`RunnableParallel`을 활용해 요약 작업과 감정 분석 작업을 **동시에 실행**합니다. 두 작업을 직렬로 호출했을 때 대비 전체 응답 시간을 줄일 수 있습니다.

In [35]:
news_text = """
오늘 코스피 지수가 개장 직후 1.2% 급락하며 약세를 보였다. 외국인 투자자 순매도가 5일 연속 이어지면서 시장의 우려가 커지고 있다.
반면 일부 반도체 관련주는 AI 수요 기대감으로 강세를 보였다. 전문가들은 단기 변동성에 대비한 분산 투자를 권고하고 있다.
"""

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 뉴스를 정확히 3줄로 요약하세요. 각 줄은 핵심 사실 1개만 담습니다."),
    ("human", "{news}"),
])

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 뉴스의 시장 감정을 한 단어로 분류: 긍정/부정/중립. 그리고 그 이유를 한 문장으로."),
    ("human", "{news}"),
])

# 두 체인을 병렬 실행
analyze = RunnableParallel(
    summary=summary_prompt | llm | parser,
    sentiment=sentiment_prompt | llm | parser,
)

result = analyze.invoke({"news": news_text})
print("📋 [요약]")
print(result["summary"])
print("\n📊 [감정 분석]")
print(result["sentiment"])

📋 [요약]
다음은 뉴스를 요약한 3줄입니다.

* 오늘 코스피 지수는 개장 직후 1.2% 급락하며 약세를 보였다.
* 외국인 투자자의 순매도가 5일 연속 이어지고 있다.
* 반도체 관련주는 AI 수요 기대감으로 강세를 보였다.

📊 [감정 분석]
부정. 시장의 약세와 외국인 투자자의 순매도 지속으로 인해 투자자들의 우려가 커지고 있다는 내용이 보도되었기 때문입니다.


---

## 실습 과제: 번역 + 톤 변환 체인

**요구사항**

1. `ChatPromptTemplate`으로 "입력된 한국어 문장을 영어로 번역한 뒤, 지정된 톤({tone}: formal/casual/polite)으로 다듬으세요"라는 프롬프트를 정의합니다.
2. `prompt | llm | parser` 체인으로 실행합니다.
3. 동일 문장을 세 가지 톤으로 `batch` 호출하여 결과를 비교합니다.

In [36]:
# 여기에 코드를 작성하세요
# Hint: ChatPromptTemplate.from_messages + chain = prompt | llm | parser + chain.batch([...])

tone_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the Korean sentence to English and rewrite it in a {tone} tone. Output English only."),
    ("human", "{text}"),
])

chain = tone_prompt | llm | parser

ko = "회의에 못 갈 것 같아요. 미안합니다."
results = chain.batch([
    {"text": ko, "tone": "formal"},
    {"text": ko, "tone": "casual"},
    {"text": ko, "tone": "polite"},
])
for tone, r in zip(["formal", "casual", "polite"], results):

[formal] I apologize, but I do not think I will be able to attend the meeting.
[casual] I'm probably going to miss the meeting, sorry about that.
[polite] I'm afraid I won't be able to attend the meeting. I apologize for the inconvenience.


<details>
<summary>모범 답안 보기</summary>

```python
tone_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the Korean sentence to English and rewrite it in a {tone} tone. Output English only."),
    ("human", "{text}"),
])

chain = tone_prompt | llm | parser

ko = "회의에 못 갈 것 같아요. 미안합니다."
results = chain.batch([
    {"text": ko, "tone": "formal"},
    {"text": ko, "tone": "casual"},
    {"text": ko, "tone": "polite"},
])
for tone, r in zip(["formal", "casual", "polite"], results):
    print(f"[{tone}] {r}")
```

</details>

---

## 트러블슈팅

| 증상 | 원인 | 해결 방법 |
|---|---|---|
| `ImportError: init_chat_model` | LangChain v0.x 사용 중 | `pip install -U langchain langchain-core` |
| `ModuleNotFoundError: langchain_google_genai` | 통합 패키지 미설치 | `pip install langchain-google-genai` |
| `RateLimitError: 429` (Gemini 무료) | 분당 15회 한도 초과 | 30초 대기 또는 Groq로 전환 |
| `chain.batch`의 응답이 느림 | 동기 실행 사용 중 | `chain.abatch()` + `asyncio.run()` 으로 비동기 처리 |
| `RunnableParallel` 결과가 dict가 아님 | 입력이 dict 형식이 아님 | `{"key": value}` 형식으로 `invoke` 호출 |

---

## 마무리

본 노트북에서 학습한 내용은 다음과 같습니다.

- `init_chat_model("provider:model")`을 통한 모델 통합 초기화
- Messages → Prompt Template → Output Parser의 표준 처리 흐름
- LCEL `|` 파이프 연산자를 통한 체인 선언
- `.invoke()`, `.stream()`, `.batch()`의 공통 지원
- `RunnableParallel`을 통한 병렬 실행, `RunnableLambda`를 통한 함수 삽입
- 한국어 비즈니스 시나리오(메뉴 추천, 뉴스 분석) 적용 사례

### 다음 노트북(03번) 예고

본 노트북에서 사용한 프롬프트는 단일 텍스트 형태였습니다. 실무에서는 다음과 같은 정밀한 통제가 필요합니다.

- "이러한 예시처럼 응답해 달라" — **Few-shot 프롬프팅**
- "정해진 JSON 형식으로 응답해 달라" — **Pydantic 기반 구조화 출력**

다음 노트북에서는 LLM 응답의 패턴과 형식을 정밀하게 제어하는 방법을 다룹니다.